In [1]:
# 필요한 라이브러리 설치 (Colab에서 처음 실행 시)
!pip install openpyxl

# 라이브러리 임포트
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("✓ 라이브러리 임포트 완료")

✓ 라이브러리 임포트 완료


In [2]:
# 파일 읽기
file_name = 'For demographic_KD.xlsx'  # 업로드한 파일명
df = pd.read_excel(file_name, sheet_name='Combined Cohorts (n = 227)')

print(f"✓ 데이터 로드 완료")
print(f"  - 총 행 수: {len(df)}")
print(f"  - 총 열 수: {len(df.columns)}")
print(f"\n컬럼 목록 (처음 20개):")
print(df.columns[:20].tolist())

✓ 데이터 로드 완료
  - 총 행 수: 988
  - 총 열 수: 172

컬럼 목록 (처음 20개):
['study_id', 'cohort_num', 'redcap_event_name', 'dem_type_visit', 'UW_visit_num', 'assay_baseline', 'age', 'sex', 'ethnic', 'race___0', 'race___1', 'race___2', 'race___3', 'race___4', 'race___5', 'race___6', 'race___7', 'race___8', 'race___9', 'race_other']


In [3]:
# Baseline 데이터만 선택
df_baseline = df[df['assay_baseline'] == 1].copy()

print(f"✓ Baseline 데이터 필터링 완료")
print(f"  - Baseline 샘플 수: {len(df_baseline)}")
print(f"\n코호트별 분포:")
print(df_baseline['cohort_num'].value_counts().sort_index())

✓ Baseline 데이터 필터링 완료
  - Baseline 샘플 수: 227

코호트별 분포:
cohort_num
1    113
2    114
Name: count, dtype: int64


In [4]:
# Step 4: 코호트별 데이터 분리

# 코호트 1과 코호트 2로 분리
cohort1 = df_baseline[df_baseline['cohort_num'] == 1].copy()
cohort2 = df_baseline[df_baseline['cohort_num'] == 2].copy()

print("=" * 80)
print("코호트별 데이터 분리 완료")
print("=" * 80)

print(f"\n코호트 1:")
print(f"  샘플 수: {len(cohort1)}")
print(f"  인지 그룹 분포:")
print(cohort1['clin_cog_group_assay_analysis'].value_counts())

print(f"\n\n코호트 2:")
print(f"  샘플 수: {len(cohort2)}")
print(f"  인지 그룹 분포:")
print(cohort2['clin_cog_group_assay_analysis'].value_counts())

# 각 코호트의 기본 정보 확인
print("\n" + "=" * 80)
print("코호트별 기본 변수 확인")
print("=" * 80)

print("\n코호트 1:")
print(f"  나이: {cohort1['age'].mean():.1f} ± {cohort1['age'].std():.1f}")
print(f"  성별 (Male %): {(cohort1['sex'] == 1).sum() / cohort1['sex'].notna().sum() * 100:.1f}%")

print("\n코호트 2:")
print(f"  나이: {cohort2['age'].mean():.1f} ± {cohort2['age'].std():.1f}")
print(f"  성별 (Male %): {(cohort2['sex'] == 1).sum() / cohort2['sex'].notna().sum() * 100:.1f}%")

코호트별 데이터 분리 완료

코호트 1:
  샘플 수: 113
  인지 그룹 분포:
clin_cog_group_assay_analysis
PD-MCI     42
Control    32
PD-NC      26
PD-D       13
Name: count, dtype: int64


코호트 2:
  샘플 수: 114
  인지 그룹 분포:
clin_cog_group_assay_analysis
PD-MCI        56
PD-NC         21
PD-D          20
Control-NC    17
Name: count, dtype: int64

코호트별 기본 변수 확인

코호트 1:
  나이: 66.1 ± 8.3
  성별 (Male %): 58.4%

코호트 2:
  나이: 65.9 ± 9.8
  성별 (Male %): 74.6%


In [5]:
# Step 5: clin_cog_group_assay_analysis 기준으로 그룹 분류

print("=" * 80)
print("Step 5: 그룹 분류 (clin_cog_group_assay_analysis 기반)")
print("=" * 80)

def classify_group_assay(row):
    """
    clin_cog_group_assay_analysis 기준 4개 그룹 분류
    - Control: Control 또는 Control-NC
    - PD-NC: PD-NC (PD with Normal Cognition)
    - PD-MCI: PD-MCI (PD with Mild Cognitive Impairment)
    - PDD: PD-D (PD with Dementia)
    """
    cog_group = row['clin_cog_group_assay_analysis']

    if cog_group in ['Control', 'Control-NC']:
        return 'Control'
    elif cog_group == 'PD-NC':
        return 'PD-NC'
    elif cog_group == 'PD-MCI':
        return 'PD-MCI'
    elif cog_group == 'PD-D':
        return 'PDD'
    else:
        return 'Unknown'

# PD 전체 vs Control
def classify_pd_control(row):
    """PD (모든 PD 환자) vs Control"""
    cog_group = row['clin_cog_group_assay_analysis']
    if cog_group in ['Control', 'Control-NC']:
        return 'Control'
    else:
        return 'PD'

# 전체 baseline 데이터에 적용
df_baseline['group_4'] = df_baseline.apply(classify_group_assay, axis=1)
df_baseline['group_pd'] = df_baseline.apply(classify_pd_control, axis=1)

# 코호트별로도 적용
cohort1['group_4'] = cohort1.apply(classify_group_assay, axis=1)
cohort1['group_pd'] = cohort1.apply(classify_pd_control, axis=1)

cohort2['group_4'] = cohort2.apply(classify_group_assay, axis=1)
cohort2['group_pd'] = cohort2.apply(classify_pd_control, axis=1)

# 통합 코호트 생성
combined = df_baseline.copy()
combined['group_4'] = combined.apply(classify_group_assay, axis=1)
combined['group_pd'] = combined.apply(classify_pd_control, axis=1)

# 결과 확인
print("\n[전체 Baseline]")
print(f"총 {len(df_baseline)}명")
print(df_baseline['group_4'].value_counts())

print("\n\n[코호트 1]")
print(f"총 {len(cohort1)}명")
print(cohort1['group_4'].value_counts())
print(f"\nPD vs Control:")
print(cohort1['group_pd'].value_counts())

print("\n\n[코호트 2]")
print(f"총 {len(cohort2)}명")
print(cohort2['group_4'].value_counts())
print(f"\nPD vs Control:")
print(cohort2['group_pd'].value_counts())

print("\n\n[통합 코호트]")
print(f"총 {len(combined)}명")
print(combined['group_4'].value_counts())
print(f"\nPD vs Control:")
print(combined['group_pd'].value_counts())

print("\n✓ 그룹 분류 완료")

Step 5: 그룹 분류 (clin_cog_group_assay_analysis 기반)

[전체 Baseline]
총 227명
group_4
PD-MCI     98
Control    49
PD-NC      47
PDD        33
Name: count, dtype: int64


[코호트 1]
총 113명
group_4
PD-MCI     42
Control    32
PD-NC      26
PDD        13
Name: count, dtype: int64

PD vs Control:
group_pd
PD         81
Control    32
Name: count, dtype: int64


[코호트 2]
총 114명
group_4
PD-MCI     56
PD-NC      21
PDD        20
Control    17
Name: count, dtype: int64

PD vs Control:
group_pd
PD         97
Control    17
Name: count, dtype: int64


[통합 코호트]
총 227명
group_4
PD-MCI     98
Control    49
PD-NC      47
PDD        33
Name: count, dtype: int64

PD vs Control:
group_pd
PD         178
Control     49
Name: count, dtype: int64

✓ 그룹 분류 완료


In [6]:
# Step 6: Missing Code 처리 및 파생변수 생성

print("=" * 80)
print("Step 6: 데이터 전처리")
print("=" * 80)

# 6-1. Missing codes를 NaN으로 변환
print("\n[6-1] Missing codes 처리 중...")
missing_codes = [-888, -802, -803, -999, -801]

for cohort_name, cohort_data in [('코호트 1', cohort1), ('코호트 2', cohort2), ('통합', combined)]:
    print(f"\n{cohort_name}:")
    count = 0
    for col in cohort_data.columns:
        if cohort_data[col].dtype in ['float64', 'int64']:
            n_missing_codes = cohort_data[col].isin(missing_codes).sum()
            if n_missing_codes > 0:
                print(f"  - {col}: {n_missing_codes}개의 missing code 변환")
                cohort_data[col] = cohort_data[col].replace(missing_codes, np.nan)
                count += 1
    if count == 0:
        print(f"  Missing code 없음")

print("\n✓ Missing codes 처리 완료")

# 6-2. RBD Symptomatology 계산
print("\n[6-2] RBD Symptomatology 계산 중...")

def calculate_rbd(row):
    """
    RBD 관련 6개 변수 중 하나라도 1 이상이면 RBD positive (1)
    모두 0이면 RBD negative (0)
    모두 missing이면 NaN
    """
    rbd_cols = ['rbd_act_drm', 'rbd_pt_injur', 'rbd_hurt_partnr',
                'rbd_sudden_move', 'rbd_drms_behav', 'rbd_jerk']

    rbd_values = [row[col] for col in rbd_cols if pd.notna(row[col])]

    if len(rbd_values) == 0:
        return np.nan

    return 1 if any(val >= 1 for val in rbd_values) else 0

cohort1['rbd_positive'] = cohort1.apply(calculate_rbd, axis=1)
cohort2['rbd_positive'] = cohort2.apply(calculate_rbd, axis=1)
combined['rbd_positive'] = combined.apply(calculate_rbd, axis=1)

print(f"  코호트 1: RBD positive={int((cohort1['rbd_positive']==1).sum())}, "
      f"negative={int((cohort1['rbd_positive']==0).sum())}, "
      f"missing={int(cohort1['rbd_positive'].isna().sum())}")
print(f"  코호트 2: RBD positive={int((cohort2['rbd_positive']==1).sum())}, "
      f"negative={int((cohort2['rbd_positive']==0).sum())}, "
      f"missing={int(cohort2['rbd_positive'].isna().sum())}")
print(f"  통합: RBD positive={int((combined['rbd_positive']==1).sum())}, "
      f"negative={int((combined['rbd_positive']==0).sum())}, "
      f"missing={int(combined['rbd_positive'].isna().sum())}")

print("✓ RBD Symptomatology 계산 완료")

# 6-3. Disease Duration 계산
print("\n[6-3] Disease Duration 계산 중...")

# 코호트 1: nexam_month (개월) → 년으로 변환
cohort1['disease_duration_years'] = cohort1['nexam_month'] / 12
n_c1_duration = cohort1['disease_duration_years'].notna().sum()
print(f"  코호트 1: {n_c1_duration}명 계산 완료 (nexam_month 사용)")

# 코호트 2: 현재 나이 - 진단 시 나이
cohort2['disease_duration_years'] = cohort2['age'] - cohort2['age_diag']
n_c2_duration_before = cohort2['disease_duration_years'].notna().sum()

# 음수 disease duration 확인 및 NaN 처리
if (cohort2['disease_duration_years'] < 0).any():
    n_negative = (cohort2['disease_duration_years'] < 0).sum()
    print(f"  ⚠️  코호트 2: {n_negative}개 음수값 발견 → NaN 처리")
    cohort2.loc[cohort2['disease_duration_years'] < 0, 'disease_duration_years'] = np.nan

n_c2_duration = cohort2['disease_duration_years'].notna().sum()
print(f"  코호트 2: {n_c2_duration}명 계산 완료 (age - age_diag 사용)")

# 통합: 코호트별 방법 적용
combined['disease_duration_years'] = np.nan
c1_mask = combined['cohort_num'] == 1
c2_mask = combined['cohort_num'] == 2

combined.loc[c1_mask, 'disease_duration_years'] = combined.loc[c1_mask, 'nexam_month'] / 12
combined.loc[c2_mask, 'disease_duration_years'] = (
    combined.loc[c2_mask, 'age'] - combined.loc[c2_mask, 'age_diag']
)

# 통합에서도 음수 처리
if (combined['disease_duration_years'] < 0).any():
    combined.loc[combined['disease_duration_years'] < 0, 'disease_duration_years'] = np.nan

n_combined_duration = combined['disease_duration_years'].notna().sum()
print(f"  통합: {n_combined_duration}명 계산 완료")

print("✓ Disease Duration 계산 완료")

# 6-4. Education 변수 정리
print("\n[6-4] Education 변수 정리...")
# 코호트 1: education 사용
# 코호트 2: education_washu 사용
cohort1['education_final'] = cohort1['education']
cohort2['education_final'] = cohort2['education_washu']

# 통합: 코호트별 적절한 변수 사용
combined['education_final'] = np.nan
combined.loc[c1_mask, 'education_final'] = combined.loc[c1_mask, 'education']
combined.loc[c2_mask, 'education_final'] = combined.loc[c2_mask, 'education_washu']

print(f"  코호트 1: education 사용 ({cohort1['education_final'].notna().sum()}/{len(cohort1)} available)")
print(f"  코호트 2: education_washu 사용 ({cohort2['education_final'].notna().sum()}/{len(cohort2)} available)")
print(f"  통합: {combined['education_final'].notna().sum()}/{len(combined)} available")

print("✓ Education 변수 정리 완료")

# 전처리 완료 요약
print("\n\n" + "=" * 80)
print("데이터 전처리 완료 요약")
print("=" * 80)

for cohort_name, cohort_data in [('코호트 1', cohort1), ('코호트 2', cohort2), ('통합', combined)]:
    print(f"\n{cohort_name} (n={len(cohort_data)})")
    for group in ['Control', 'PD-NC', 'PD-MCI', 'PDD']:
        n = len(cohort_data[cohort_data['group_4'] == group])
        print(f"  - {group}: {n}명")

print("\n✓ Step 6 완료. 다음 단계: 통계 함수 정의")

Step 6: 데이터 전처리

[6-1] Missing codes 처리 중...

코호트 1:
  Missing code 없음

코호트 2:
  - moca_total: 2개의 missing code 변환
  - updrs_i_score: 7개의 missing code 변환
  - updrs_ii_score: 5개의 missing code 변환
  - updrs_iii_score: 2개의 missing code 변환
  - updrs_iv_score: 12개의 missing code 변환
  - updrs_tot_score: 17개의 missing code 변환
  - hoehn_yahr: 4개의 missing code 변환

통합:
  - moca_total: 2개의 missing code 변환
  - updrs_i_score: 7개의 missing code 변환
  - updrs_ii_score: 5개의 missing code 변환
  - updrs_iii_score: 2개의 missing code 변환
  - updrs_iv_score: 12개의 missing code 변환
  - updrs_tot_score: 17개의 missing code 변환
  - hoehn_yahr: 4개의 missing code 변환

✓ Missing codes 처리 완료

[6-2] RBD Symptomatology 계산 중...
  코호트 1: RBD positive=49, negative=57, missing=7
  코호트 2: RBD positive=33, negative=18, missing=63
  통합: RBD positive=82, negative=75, missing=70
✓ RBD Symptomatology 계산 완료

[6-3] Disease Duration 계산 중...
  코호트 1: 80명 계산 완료 (nexam_month 사용)
  코호트 2: 96명 계산 완료 (age - age_diag 사용)
  통합: 176명 계산 완료
✓ Disease Du

In [7]:
# Disease Duration 계산: age - age_diag (HC는 0으로 설정)
import numpy as np

for cohort_name, cohort_data in [('코호트 1', cohort1), ('코호트 2', cohort2), ('통합', combined)]:
    # PD 환자: age - age_diag
    cohort_data['disease_duration'] = np.where(
        cohort_data['age_diag'].notna() & (cohort_data['age_diag'] > 0),
        cohort_data['age'] - cohort_data['age_diag'],
        np.nan
    )
    # HC (Control): duration = 0
    cohort_data.loc[cohort_data['group_4'] == 'Control', 'disease_duration'] = 0

    valid = cohort_data['disease_duration'].notna().sum()
    total = len(cohort_data)
    print(f"{cohort_name}: disease_duration 계산 완료 ({valid}/{total}명)")

코호트 1: disease_duration 계산 완료 (113/113명)
코호트 2: disease_duration 계산 완료 (113/114명)
통합: disease_duration 계산 완료 (226/227명)


In [8]:
# Disease Duration 계산: age - age_diag (HC는 0으로 설정)
import numpy as np

for cohort_name, cohort_data in [('코호트 1', cohort1), ('코호트 2', cohort2), ('통합', combined)]:
    # PD 환자: age - age_diag
    cohort_data['disease_duration_years'] = np.where(
        cohort_data['age_diag'].notna() & (cohort_data['age_diag'] > 0),
        cohort_data['age'] - cohort_data['age_diag'],
        np.nan
    )
    # HC (Control): duration = 0
    cohort_data.loc[cohort_data['group_4'] == 'Control', 'disease_duration_years'] = 0

    valid = cohort_data['disease_duration_years'].notna().sum()
    total = len(cohort_data)
    print(f"{cohort_name}: disease_duration_years 계산 완료 ({valid}/{total}명)")

코호트 1: disease_duration_years 계산 완료 (113/113명)
코호트 2: disease_duration_years 계산 완료 (113/114명)
통합: disease_duration_years 계산 완료 (226/227명)


In [9]:
# Step 7: 통계 함수 정의 (Kruskal-Wallis & Chi-square)

print("=" * 80)
print("Step 7: 통계 함수 정의")
print("=" * 80)

from scipy import stats

# 7-1. 기술통계 함수 (실제 N 포함)
def calc_mean_sd_with_n(data, var):
    """
    평균(표준편차) 계산 및 실제 N 반환
    Returns: (결과 문자열, 실제 N)
    """
    valid_data = data[var].dropna()
    n = len(valid_data)

    if n == 0:
        return "n/a", 0

    mean = valid_data.mean()
    sd = valid_data.std()
    return f"{mean:.2f} ({sd:.2f})", n

def calc_percentage_with_n(data, var, value=1):
    """
    백분율 계산 및 실제 N 반환
    Returns: (결과 문자열, 실제 N)
    """
    valid_data = data[var].dropna()
    n = len(valid_data)

    if n == 0:
        return "n/a", 0

    count = (valid_data == value).sum()
    pct = (count / n) * 100
    return f"{pct:.0f}", n

# 7-2. 통계 검정 함수
def perform_kruskal_wallis(df_cohort, var):
    """
    Kruskal-Wallis H test for continuous/ordinal variables
    4개 그룹 간 비교: Control, PD-NC, PD-MCI, PDD
    """
    groups = ['Control', 'PD-NC', 'PD-MCI', 'PDD']

    # 각 그룹의 데이터 추출 (결측치 제외)
    group_data = []
    for g in groups:
        data = df_cohort[df_cohort['group_4'] == g][var].dropna()
        if len(data) > 0:
            group_data.append(data)

    # 최소 2개 이상의 그룹에 데이터가 있어야 검정 가능
    if len(group_data) < 2:
        return np.nan, "insufficient data"

    try:
        h_stat, p_value = stats.kruskal(*group_data)
        return p_value, "Kruskal-Wallis"
    except Exception as e:
        return np.nan, f"error"

def perform_chi_square(df_cohort, var):
    """
    Pearson Chi-square test for categorical variables
    Expected frequency < 5인 경우 경고 출력
    """
    try:
        # 결측치 제거
        valid_data = df_cohort[['group_4', var]].dropna()

        if len(valid_data) < 10:
            return np.nan, "insufficient data"

        # Contingency table 생성
        contingency = pd.crosstab(valid_data['group_4'], valid_data[var])

        # Chi-square test
        chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

        # Expected frequency 체크
        min_expected = expected.min()
        if min_expected < 5:
            warning_msg = f"Chi-square (⚠️ min expected={min_expected:.1f}<5)"
            return p_value, warning_msg

        return p_value, "Chi-square"

    except Exception as e:
        return np.nan, f"error"

def perform_statistical_test(df_cohort, var, var_type='continuous'):
    """
    변수 타입에 따라 적절한 통계 검정 수행

    Parameters:
    - var_type: 'continuous', 'ordinal', 'categorical'

    Returns:
    - p_value, test_name
    """
    if var_type in ['continuous', 'ordinal']:
        return perform_kruskal_wallis(df_cohort, var)
    elif var_type == 'categorical':
        return perform_chi_square(df_cohort, var)
    else:
        return np.nan, "unknown type"

def format_pvalue(p):
    """p-value 포맷팅"""
    if pd.isna(p):
        return "n/a"
    elif p < 0.001:
        return "<0.001"
    elif p < 0.01:
        return "<0.01"
    else:
        return f"{p:.2f}"

# 7-3. 변수 타입 정의
variable_types = {
    # Continuous variables
    'age': 'continuous',
    'education_final': 'continuous',
    'tot_led': 'continuous',
    'moca_total': 'continuous',
    'updrs_i_score': 'continuous',
    'updrs_ii_score': 'continuous',
    'updrs_iii_score': 'continuous',
    'updrs_iv_score': 'continuous',
    'updrs_tot_score': 'continuous',
    'hamd_total': 'continuous',
    'hama_total': 'continuous',
    'upsit_score': 'continuous',
    'disease_duration_years': 'continuous',

    # Ordinal variable
    'hoehn_yahr': 'ordinal',

    # Categorical variables (binary)
    'sex': 'categorical',
    'race___0': 'categorical',
    'rbd_positive': 'categorical'
}

print("\n변수 타입 정의:")
print(f"  - Continuous/Ordinal (Kruskal-Wallis): {sum(1 for v in variable_types.values() if v in ['continuous', 'ordinal'])}")
print(f"  - Categorical (Chi-square): {sum(1 for v in variable_types.values() if v == 'categorical')}")

# 7-4. 테스트: 몇 가지 변수로 함수 테스트
print("\n\n" + "=" * 80)
print("통계 함수 테스트")
print("=" * 80)

test_vars = [
    ('age', 'continuous'),
    ('sex', 'categorical'),
    ('moca_total', 'continuous')
]

print("\n코호트 1 테스트:")
for var, var_type in test_vars:
    if var in cohort1.columns:
        # Control 그룹 예시
        control_data = cohort1[cohort1['group_4'] == 'Control']
        val, n = calc_mean_sd_with_n(control_data, var) if var_type == 'continuous' else calc_percentage_with_n(control_data, var)

        # p-value
        p_val, test_name = perform_statistical_test(cohort1, var, var_type)

        print(f"  {var}: Control={val} [n={n}], p={format_pvalue(p_val)} [{test_name}]")

print("\n✓ Step 7 완료!")
print("  - Kruskal-Wallis H test: continuous & ordinal 변수")
print("  - Pearson Chi-square test: categorical 변수")

Step 7: 통계 함수 정의

변수 타입 정의:
  - Continuous/Ordinal (Kruskal-Wallis): 14
  - Categorical (Chi-square): 3


통계 함수 테스트

코호트 1 테스트:
  age: Control=67.09 (9.00) [n=32], p=<0.001 [Kruskal-Wallis]
  sex: Control=31 [n=32], p=<0.001 [Chi-square]
  moca_total: Control=27.81 (1.31) [n=32], p=<0.001 [Kruskal-Wallis]

✓ Step 7 완료!
  - Kruskal-Wallis H test: continuous & ordinal 변수
  - Pearson Chi-square test: categorical 변수


In [10]:
# Step 8: Hoehn & Yahr 처리 및 테이블 생성 함수

print("=" * 80)
print("Step 8: Hoehn & Yahr 처리 및 테이블 생성 함수 정의")
print("=" * 80)

# 8-1. Hoehn & Yahr 실제 Stage 확인
print("\n[8-1] Hoehn & Yahr Stage 확인")

print("\n코호트 1 (PD 환자만):")
pd1_hy = cohort1[cohort1['group_pd'] == 'PD']['hoehn_yahr']
pd1_stages = sorted([s for s in pd1_hy.dropna().unique() if s > 0])
print(f"  Stage: {pd1_stages}")

print("\n코호트 2 (PD 환자만):")
pd2_hy = cohort2[cohort2['group_pd'] == 'PD']['hoehn_yahr']
pd2_stages = sorted([s for s in pd2_hy.dropna().unique() if s > 0])
print(f"  Stage: {pd2_stages}")

print("\n통합 (PD 환자만):")
combined_hy = combined[combined['group_pd'] == 'PD']['hoehn_yahr']
combined_stages = sorted([s for s in combined_hy.dropna().unique() if s > 0])
print(f"  Stage: {combined_stages}")

# 8-2. Hoehn & Yahr 추가 함수
def add_hoehn_yahr_row(cohort_data, results, n_tracker, col_names, stages):
    """
    Hoehn & Yahr Stage별 빈도 추가
    """
    var_name = 'hoehn_yahr'

    if var_name not in cohort_data.columns:
        return

    # 그룹별 데이터
    control = cohort_data[cohort_data['group_4'] == 'Control']
    pd_all = cohort_data[cohort_data['group_pd'] == 'PD']
    pd_nc = cohort_data[cohort_data['group_4'] == 'PD-NC']
    pd_mci = cohort_data[cohort_data['group_4'] == 'PD-MCI']
    pdd = cohort_data[cohort_data['group_4'] == 'PDD']

    col_pd, col_control, col_pd_nc, col_pd_mci, col_pdd = col_names

    # Stage 라벨 생성
    stage_labels = [f"Stage {int(s)}" if s % 1 == 0 else f"Stage {s:.1f}"
                    for s in stages]

    def format_hy_counts(data, stages):
        """각 stage별 빈도를 문자열로 반환"""
        if len(data[var_name].dropna()) == 0:
            return "n/a"

        counts = []
        for stage in stages:
            count = (data[var_name] == stage).sum()
            counts.append(str(int(count)))

        return " / ".join([f"({c})" for c in counts])

    # 각 그룹별 계산
    val_pd = format_hy_counts(pd_all, stages)
    val_control = "n/a"
    val_pd_nc = format_hy_counts(pd_nc, stages)
    val_pd_mci = format_hy_counts(pd_mci, stages)
    val_pdd = format_hy_counts(pdd, stages)

    # 통계 검정 (PD 환자만, Stage > 0인 경우만)
    pd_only = cohort_data[cohort_data['group_pd'] == 'PD']
    pd_only_valid = pd_only[pd_only[var_name] > 0].copy()
    p_val, test_name = perform_statistical_test(pd_only_valid, var_name, 'ordinal')

    # N 계산 (Stage > 0인 경우만)
    n_pd = (pd_all[var_name] > 0).sum()
    n_control = 0
    n_pd_nc = (pd_nc[var_name] > 0).sum()
    n_pd_mci = (pd_mci[var_name] > 0).sum()
    n_pdd = (pdd[var_name] > 0).sum()

    # Variable label 생성
    stage_label_str = " / ".join(stage_labels)
    variable_label = f"Hoehn and Yahr\n  {stage_label_str}"

    # 결과 저장
    results.append({
        'Variable': variable_label,
        col_pd: val_pd,
        col_control: val_control,
        col_pd_nc: val_pd_nc,
        col_pd_mci: val_pd_mci,
        col_pdd: val_pdd,
        'p-value': format_pvalue(p_val)
    })

    n_tracker.append({
        'Variable': f'Hoehn and Yahr ({stage_label_str})',
        'PD_n': n_pd,
        'Control_n': n_control,
        'PD-NC_n': n_pd_nc,
        'PD-MCI_n': n_pd_mci,
        'PDD_n': n_pdd,
        'Test': test_name
    })

print("\n✓ Hoehn & Yahr 함수 정의 완료")

# 8-3. 메인 테이블 생성 함수
def create_demographic_table(cohort_data, cohort_name, include_hamilton_upsit=False):
    """
    Demographic table 생성

    Parameters:
    - cohort_data: 데이터프레임
    - cohort_name: 'Cohort 1', 'Cohort 2', 'Combined'
    - include_hamilton_upsit: Hamilton/UPSIT 포함 여부
    """

    # 그룹별 데이터 분리
    control = cohort_data[cohort_data['group_4'] == 'Control']
    pd_all = cohort_data[cohort_data['group_pd'] == 'PD']
    pd_nc = cohort_data[cohort_data['group_4'] == 'PD-NC']
    pd_mci = cohort_data[cohort_data['group_4'] == 'PD-MCI']
    pdd = cohort_data[cohort_data['group_4'] == 'PDD']

    results = []
    n_tracker = []

    col_pd = f'PD (n={len(pd_all)})'
    col_control = f'Control (n={len(control)})'
    col_pd_nc = f'PD-NC (n={len(pd_nc)})'
    col_pd_mci = f'PD-MCI (n={len(pd_mci)})'
    col_pdd = f'PDD (n={len(pdd)})'
    col_names = (col_pd, col_control, col_pd_nc, col_pd_mci, col_pdd)

    def add_continuous_variable(var_name, label, test_type='continuous', pd_only_test=False):
        """연속형 변수 추가"""
        if var_name not in cohort_data.columns:
            return

        val_pd, n_pd = calc_mean_sd_with_n(pd_all, var_name)
        val_control, n_control = calc_mean_sd_with_n(control, var_name) if not pd_only_test else ("n/a", 0)
        val_pd_nc, n_pd_nc = calc_mean_sd_with_n(pd_nc, var_name)
        val_pd_mci, n_pd_mci = calc_mean_sd_with_n(pd_mci, var_name)
        val_pdd, n_pdd = calc_mean_sd_with_n(pdd, var_name)

        test_data = cohort_data[cohort_data['group_pd'] == 'PD'] if pd_only_test else cohort_data
        p_val, test_name = perform_statistical_test(test_data, var_name, test_type)

        results.append({
            'Variable': label,
            col_pd: val_pd,
            col_control: val_control,
            col_pd_nc: val_pd_nc,
            col_pd_mci: val_pd_mci,
            col_pdd: val_pdd,
            'p-value': format_pvalue(p_val)
        })

        n_tracker.append({
            'Variable': label,
            'PD_n': n_pd,
            'Control_n': n_control,
            'PD-NC_n': n_pd_nc,
            'PD-MCI_n': n_pd_mci,
            'PDD_n': n_pdd,
            'Test': test_name
        })

    def add_categorical_variable(var_name, label, value=1):
        """카테고리형 변수 추가"""
        if var_name not in cohort_data.columns:
            return

        val_pd, n_pd = calc_percentage_with_n(pd_all, var_name, value)
        val_control, n_control = calc_percentage_with_n(control, var_name, value)
        val_pd_nc, n_pd_nc = calc_percentage_with_n(pd_nc, var_name, value)
        val_pd_mci, n_pd_mci = calc_percentage_with_n(pd_mci, var_name, value)
        val_pdd, n_pdd = calc_percentage_with_n(pdd, var_name, value)

        p_val, test_name = perform_statistical_test(cohort_data, var_name, 'categorical')

        results.append({
            'Variable': label,
            col_pd: val_pd,
            col_control: val_control,
            col_pd_nc: val_pd_nc,
            col_pd_mci: val_pd_mci,
            col_pdd: val_pdd,
            'p-value': format_pvalue(p_val)
        })

        n_tracker.append({
            'Variable': label,
            'PD_n': n_pd,
            'Control_n': n_control,
            'PD-NC_n': n_pd_nc,
            'PD-MCI_n': n_pd_mci,
            'PDD_n': n_pdd,
            'Test': test_name
        })

    # ========== 변수 추가 ==========

    add_continuous_variable('age', 'Age, years (SD)', 'continuous')
    add_categorical_variable('sex', 'Gender, % male', value=1)
    add_categorical_variable('race___0', 'Race, % White', value=1)
    add_continuous_variable('education_final', 'Education, years (SD)', 'continuous')
    add_continuous_variable('tot_led', 'Levodopa Equivalent daily dose, mg (SD)', 'continuous', pd_only_test=True)
    add_continuous_variable('moca_total', 'MoCA total score (SD)', 'continuous')
    add_continuous_variable('updrs_i_score', 'MDS-UPDRS part I', 'continuous')
    add_continuous_variable('updrs_ii_score', 'MDS-UPDRS part II', 'continuous')
    add_continuous_variable('updrs_iii_score', 'MDS-UPDRS part III', 'continuous')
    add_continuous_variable('updrs_iv_score', 'MDS-UPDRS part IV', 'continuous')
    add_continuous_variable('updrs_tot_score', 'MDS-UPDRS Total', 'continuous')

    # Hoehn & Yahr (코호트별 적절한 stages 사용)
    if cohort_name == 'Cohort 1':
        hy_stages = pd1_stages
    elif cohort_name == 'Cohort 2':
        hy_stages = pd2_stages
    else:  # Combined
        hy_stages = combined_stages

    add_hoehn_yahr_row(cohort_data, results, n_tracker, col_names, hy_stages)

    # Hamilton & UPSIT (코호트 1만 또는 통합에서 코호트 1 데이터만)
    if include_hamilton_upsit:
        add_continuous_variable('hamd_total', 'Hamilton Depression', 'continuous')
        add_continuous_variable('hama_total', 'Hamilton Anxiety', 'continuous')
        add_continuous_variable('upsit_score', 'UPSIT', 'continuous')

    # RBD
    add_categorical_variable('rbd_positive', 'RBD symptomatology, %', value=1)

    # Disease duration
    add_continuous_variable('disease_duration_years', 'Disease duration, years (SD)', 'continuous', pd_only_test=True)

    result_df = pd.DataFrame(results)
    n_tracker_df = pd.DataFrame(n_tracker)

    return result_df, n_tracker_df

print("\n✓ Step 8 완료!")
print("  테이블 생성 함수 준비 완료")

Step 8: Hoehn & Yahr 처리 및 테이블 생성 함수 정의

[8-1] Hoehn & Yahr Stage 확인

코호트 1 (PD 환자만):
  Stage: [np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0)]

코호트 2 (PD 환자만):
  Stage: [np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(4.0), np.float64(5.0)]

통합 (PD 환자만):
  Stage: [np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0)]

✓ Hoehn & Yahr 함수 정의 완료

✓ Step 8 완료!
  테이블 생성 함수 준비 완료


In [11]:
# Step 9: 최종 테이블 생성 및 Excel 저장

print("=" * 80)
print("Step 9: 최종 Demographic Tables 생성")
print("=" * 80)

# 9-1. 코호트 1 테이블 생성
print("\n[코호트 1 테이블 생성 중...]")
table_c1, n_tracker_c1 = create_demographic_table(
    cohort1,
    'Cohort 1',
    include_hamilton_upsit=True  # 코호트 1만 Hamilton/UPSIT 포함
)
print(f"✓ 완료: {len(table_c1)}개 변수")

# 9-2. 코호트 2 테이블 생성
print("\n[코호트 2 테이블 생성 중...]")
table_c2, n_tracker_c2 = create_demographic_table(
    cohort2,
    'Cohort 2',
    include_hamilton_upsit=False  # 코호트 2는 Hamilton/UPSIT 없음
)
print(f"✓ 완료: {len(table_c2)}개 변수")

# 9-3. 통합 테이블 생성
print("\n[통합 테이블 생성 중...]")

# 통합 테이블은 코호트 1 데이터만 있는 변수(Hamilton/UPSIT)를 포함
# 하지만 코호트 1의 샘플만 사용했다고 명시
table_combined, n_tracker_combined = create_demographic_table(
    combined,
    'Combined',
    include_hamilton_upsit=True  # 코호트 1 데이터만 사용
)
print(f"✓ 완료: {len(table_combined)}개 변수")

# 9-4. Excel 저장
print("\n\n" + "=" * 80)
print("Excel 파일 저장 중...")
print("=" * 80)

from google.colab import files
import openpyxl
from openpyxl.styles import Font, Alignment

output_filename = 'Demographic_Tables_Final.xlsx'

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    # 코호트 1
    table_c1.to_excel(writer, sheet_name='Cohort 1', index=False)
    n_tracker_c1.to_excel(writer, sheet_name='Cohort 1 - Sample N', index=False)

    # 코호트 2
    table_c2.to_excel(writer, sheet_name='Cohort 2', index=False)
    n_tracker_c2.to_excel(writer, sheet_name='Cohort 2 - Sample N', index=False)

    # 통합
    table_combined.to_excel(writer, sheet_name='Combined', index=False)
    n_tracker_combined.to_excel(writer, sheet_name='Combined - Sample N', index=False)

    # 컬럼 너비 조정
    for sheet_name in ['Cohort 1', 'Cohort 2', 'Combined']:
        worksheet = writer.sheets[sheet_name]
        worksheet.column_dimensions['A'].width = 45
        for col in ['B', 'C', 'D', 'E', 'F', 'G']:
            worksheet.column_dimensions[col].width = 20

    for sheet_name in ['Cohort 1 - Sample N', 'Cohort 2 - Sample N', 'Combined - Sample N']:
        worksheet = writer.sheets[sheet_name]
        worksheet.column_dimensions['A'].width = 45
        for col in ['B', 'C', 'D', 'E', 'F', 'G']:
            worksheet.column_dimensions[col].width = 15

print(f"\n✓ Excel 파일 생성 완료: {output_filename}")
print("\n파일 구성:")
print("  - Sheet 1: Cohort 1 Demographic Table")
print("  - Sheet 2: Cohort 1 - Sample N (각 변수별 실제 분석 샘플 수)")
print("  - Sheet 3: Cohort 2 Demographic Table")
print("  - Sheet 4: Cohort 2 - Sample N (각 변수별 실제 분석 샘플 수)")
print("  - Sheet 5: Combined Demographic Table")
print("  - Sheet 6: Combined - Sample N (각 변수별 실제 분석 샘플 수)")

# 9-5. 파일 다운로드
print("\n파일 다운로드 중...")
files.download(output_filename)

# 9-6. 미리보기
print("\n\n" + "=" * 80)
print("코호트 1 테이블 미리보기")
print("=" * 80)
print(table_c1.to_string(index=False, max_colwidth=50))

print("\n\n" + "=" * 80)
print("코호트 2 테이블 미리보기")
print("=" * 80)
print(table_c2.to_string(index=False, max_colwidth=50))

print("\n\n" + "=" * 80)
print("통합 테이블 미리보기")
print("=" * 80)
print(table_combined.to_string(index=False, max_colwidth=50))

# 9-7. 최종 요약
print("\n\n" + "=" * 80)
print("✓✓✓ 모든 작업 완료! ✓✓✓")
print("=" * 80)

print("\n생성된 테이블 특징:")
print("  ✓ 3개 테이블: 코호트 1, 코호트 2, 통합")
print("  ✓ clin_cog_group_assay_analysis 기반 그룹 분류")
print("  ✓ 코호트별 맞춤형 Hoehn & Yahr Stage")
print("  ✓ Kruskal-Wallis test (continuous/ordinal)")
print("  ✓ Chi-square test (categorical)")
print("  ✓ 각 변수별 실제 분석 N 추적")
print("  ✓ Hamilton/UPSIT: 코호트 1만 포함")
print("  ✓ Missing value 자동 제거 후 통계")

print("\n샘플 크기 요약:")
print("  코호트 1 (n=113):")
for group in ['Control', 'PD-NC', 'PD-MCI', 'PDD']:
    n = len(cohort1[cohort1['group_4'] == group])
    print(f"    - {group}: {n}명")

print("\n  코호트 2 (n=114):")
for group in ['Control', 'PD-NC', 'PD-MCI', 'PDD']:
    n = len(cohort2[cohort2['group_4'] == group])
    print(f"    - {group}: {n}명")

print("\n  통합 (n=227):")
for group in ['Control', 'PD-NC', 'PD-MCI', 'PDD']:
    n = len(combined[combined['group_4'] == group])
    print(f"    - {group}: {n}명")

print("\n" + "=" * 80)

Step 9: 최종 Demographic Tables 생성

[코호트 1 테이블 생성 중...]
✓ 완료: 17개 변수

[코호트 2 테이블 생성 중...]
✓ 완료: 14개 변수

[통합 테이블 생성 중...]
✓ 완료: 17개 변수


Excel 파일 저장 중...

✓ Excel 파일 생성 완료: Demographic_Tables_Final.xlsx

파일 구성:
  - Sheet 1: Cohort 1 Demographic Table
  - Sheet 2: Cohort 1 - Sample N (각 변수별 실제 분석 샘플 수)
  - Sheet 3: Cohort 2 Demographic Table
  - Sheet 4: Cohort 2 - Sample N (각 변수별 실제 분석 샘플 수)
  - Sheet 5: Combined Demographic Table
  - Sheet 6: Combined - Sample N (각 변수별 실제 분석 샘플 수)

파일 다운로드 중...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



코호트 1 테이블 미리보기
                                          Variable               PD (n=81) Control (n=32)           PD-NC (n=26)          PD-MCI (n=42)            PDD (n=13) p-value
                                   Age, years (SD)            65.67 (8.03)   67.09 (9.00)           60.73 (7.78)           66.14 (6.13)          74.00 (6.83)  <0.001
                                    Gender, % male                      69             31                     50                     76                    85  <0.001
                                     Race, % White                      98             88                    100                     98                    92    0.13
                             Education, years (SD)            17.89 (2.17)   18.09 (1.97)           17.88 (2.30)           17.95 (2.08)          17.69 (2.36)    0.99
           Levodopa Equivalent daily dose, mg (SD)         720.11 (474.55)            n/a        785.41 (519.17)        691.30 (434.10)       682.62 (531